[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/hydrology/topmodel_storm.ipynb)

# A storm in TOPMODEL: Ettrick Water, Southern Uplands of Scotland

This notebook runs **TOPMODEL** on every 30 m pixel of a small catchment through one storm, hour by hour, and maps how the catchment wets up: where the ground saturates, and how far the rest is from saturation. It uses the same catchment as the S-HYPE notebook in this folder: the **Ettrick Water above Brockhoperig** (NRFA gauging station 21017, 37.5 km² of the Southern Uplands). The storm is the flood of **late October 2021**, the largest in the gauged record for 2021–2024.

There is no calibration here: we run one simulation with parameter values inside their usual ranges, and look at what the model does.

The workflow is:
1. Install `lsdviztools3` (plotting) and the `lsdtt3` command-line tools.
2. Get the gauge and its daily flows from the National River Flow Archive (NRFA).
3. Download a **30 m** DEM, outline the catchment, and cut the DEM to it.
4. Download **hourly rainfall** for the storm.
5. Run TOPMODEL with `lsdtt-distributed-hydrology` (`column_model: topmodel`), one-hour steps.
6. Compare the hydrograph with the gauge, and map the saturation state at a few times during the storm.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## How TOPMODEL works

TOPMODEL (Beven and Kirkby, 1979) is built on one idea: **water drains downslope underground**, so the ground saturates first where a lot of upslope area drains through a gentle slope: valley floors, hollows, the edges of streams. Rain falling on saturated ground cannot soak in and runs straight to the stream (**saturation-excess runoff**), and the saturated area grows and shrinks with the wetness of the catchment: the **variable contributing area**.

The topography enters through the **topographic wetness index** of each pixel,

```
TWI = ln(a / tan β)
```

where `a` is the upslope area draining through the pixel per unit contour length and `tan β` the local slope. High TWI means wet.

In classic TOPMODEL (`saturated_zone_mode: index`), the whole catchment shares one mean **saturation deficit** `S̄`: the depth of water needed to saturate the soil, in metres. Each pixel's own deficit follows from its TWI,

```
D_i = S̄ + m (λ − TWI_i)
```

where `λ` is the catchment's mean TWI and `m` (`szm_metres`) sets how fast the soil's transmissivity falls with depth. A pixel with `D_i ≤ 0` is saturated. Rain that soaks in slowly reduces `S̄`, so the deficits fall everywhere and the saturated area spreads up the valleys; drainage from the saturated zone, the **subsurface flow** `T₀ e^(−λ) e^(−S̄/m)`, feeds the stream and raises `S̄` again.

`lsdtt3` computes the TWI from the DEM (D-infinity flow routing, and a slope fitted over a small window), runs the water balance on every pixel, and routes the runoff to the outlet through the travel times of the flow network.

### How this differs from S-HYPE

S-HYPE (the other notebook) divides a catchment into **sub-basins and classes** with no locations inside them, and runs daily: good for seasons and years. TOPMODEL works on **every pixel**, at any time step (hourly here): good for storms, and for **where** the catchment saturates. TOPMODEL is exactly the mechanism S-HYPE could not represent with a near-stream soil class: valley floors kept wet by water draining into them from the hillslopes. The [README](README.md) in this folder compares the two, including how long they take to run.

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3`, the plotting package, is on [PyPI](https://pypi.org/project/lsdviztools3/). It needs Python 3.12 or later, which Colab has.

In [ ]:
!pip install -q "lsdviztools3>=0.4.2"

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

In [ ]:
!lsdtt-distributed-hydrology -v

## Step 1: The gauge, from the National River Flow Archive

The NRFA's web service needs no account or key. We get station 21017's location and catchment area, and its gauged daily flows around the storm. The NRFA archives **daily mean** flows, so the gauge record cannot show the hourly peak; we compare daily means.

In [ ]:
import json
import urllib.request

import numpy as np
import pandas as pd

station = 21017
storm_start, storm_end = "2021-10-25", "2021-10-30"    # six days, starting the day before the rain

NRFA = "https://nrfaapps.ceh.ac.uk/nrfa/ws"
info = json.load(urllib.request.urlopen(
    f"{NRFA}/station-info?station={station}&format=json-object"
    "&fields=id,name,catchment-area,latitude,longitude"))["data"][0]
print(info)

series = json.load(urllib.request.urlopen(
    f"{NRFA}/time-series?station={station}&data-type=gdf&format=json-object"
    f"&start-date=2021-10-20&end-date=2021-11-02"))["data-stream"]
observed = pd.DataFrame({"date": pd.to_datetime(series[0::2]), "flow_m3s": series[1::2]})
print(observed.to_string(index=False))

pd.DataFrame({"latitude": [info["latitude"]], "longitude": [info["longitude"]],
              "point_id": [station]}).to_csv("ettrick_gauge.csv", index=False)

## Step 2: The DEM and the catchment

As in the S-HYPE notebook: a 30 m Copernicus DEM for a box around the catchment, a hillshade, and the catchment above the gauge as a polygon from `lsdtt-basin-extraction`.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (55.32, -3.36)
upper_right = (55.45, -3.15)

with open("ettrick_fetch.param", "w") as f:
    f.write("write_fname: ettrick\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
    f.write("dem_source: cop30_aws\n")
!lsdtt-fetch-raster ./ ettrick_fetch.param

with open("ettrick_hillshade.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")
!lsdtt-raster-preprocessing ./ ettrick_hillshade.param

with open("ettrick_basin.param", "w") as f:
    f.write("read_fname: ettrick_DEM.tif\n")
    f.write("write_fname: ettrick_basin\n")
    f.write("threshold_contributing_pixels: 500\n")
    f.write("select_basins_from_outlets: true\n")
    f.write("basin_outlet_fname: ettrick_gauge.csv\n")
    f.write("basin_outlet_snapping_method: expanding_square\n")
    f.write("basin_outlet_contributing_pixels_threshold: 500\n")
    f.write("write_basin_outlines: true\n")
    f.write("write_basin_outlines_as_polygon: true\n")
    f.write("basin_outlines_write_vector_format: flatgeobuf\n")
!lsdtt-basin-extraction ./ ettrick_basin.param

### Cut the DEM to the catchment

TOPMODEL runs on **every** pixel of the DEM it is given, and its discharge is everything leaving the DEM. To model only the gauged catchment, we set every pixel outside the catchment to nodata. The catchment's outlet then drains into nodata, which the model treats as its outlet.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio import features

catchment = gpd.read_file("ettrick_basin_junction_basin_polygons.fgb")
with rasterio.open("ettrick_DEM.tif") as src:
    dem = src.read(1)
    profile = src.profile
    transform = src.transform
    nodata = src.nodata

inside = features.rasterize([(geom, 1) for geom in catchment.geometry],
                            out_shape=dem.shape, transform=transform, fill=0).astype(bool)
with rasterio.open("ettrick_catchment_DEM.tif", "w", **profile) as dst:
    dst.write(np.where(inside & (dem != nodata), dem, nodata).astype(dem.dtype), 1)

catchment_area_m2 = catchment.area.sum()
print(f"Catchment: {inside.sum()} pixels, {catchment_area_m2 / 1e6:.1f} km2 (NRFA: {info['catchment-area']} km2)")

## Step 3: Hourly rainfall for the storm

From the [Open-Meteo historical weather API](https://open-meteo.com/en/docs/historical-weather-api), keyless: **hourly** precipitation at the middle of the catchment, 25 to 30 October 2021. The model reads it as a CSV with a `rainfall_mm` column, one row per time step.

We give no evaporation (`pet_fname` left out means zero): over six wet days in late October it is a few millimetres at most, against well over 100 mm of rain.

**A caution about the forcing.** The rainfall comes from a reanalysis, not a rain gauge. Compare its timing with the gauged flows in Step 1: the river was already rising on 26 October, when this series has little rain. A reanalysis can be several hours out for a single storm, and the simulated flood will inherit that.

In [ ]:
import matplotlib.pyplot as plt

url = ("https://archive-api.open-meteo.com/v1/archive?latitude=55.38&longitude=-3.27"
       f"&start_date={storm_start}&end_date={storm_end}&hourly=precipitation&timezone=GMT")
hourly = json.load(urllib.request.urlopen(url))["hourly"]
rain = pd.DataFrame({"time": pd.to_datetime(hourly["time"]), "rainfall_mm": hourly["precipitation"]})
rain.to_csv("storm_rainfall.csv", index=False)
print(f"{len(rain)} hours, {rain.rainfall_mm.sum():.0f} mm in total, wettest hour {rain.rainfall_mm.max():.1f} mm")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(rain.time, rain.rainfall_mm, width=1 / 24, color="tab:blue")
ax.set_ylabel("Rainfall (mm/h)")
ax.set_title("Hourly rainfall, Ettrick catchment, October 2021")
plt.show()

## Step 4: Parameters and the starting state

TOPMODEL needs only a few parameters. We use values inside their usual ranges, picked by hand from a few trial runs: **no calibration**.

| Parameter | Value | Meaning |
|---|---|---|
| `szm_metres` | 0.02 m | `m`: how fast transmissivity falls as the soil dries. Small values give flashy, quickly receding floods |
| `ln_t0` | 4.0 | ln of the transmissivity of the saturated soil, T₀ in m² h⁻¹ |
| `td_hours_per_metre` | 50 h m⁻¹ | how slowly water drains down through the unsaturated soil to the water table |
| `srmax_metres` | 0.05 m | the root zone's storage: water that only evaporation can remove |
| `sr0_metres` | 0.002 m | the root zone's deficit at the start: wet, as in late October |
| `twi_window_radius_metres` | 45 m | the window for the fitted slope in the TWI. Pixels within this distance of the edge of the data have no slope and are left out |

**The starting state comes from the gauge.** In classic TOPMODEL the initial mean deficit is set by the subsurface flow at the start, `q0_metres_per_step`. We take the gauged flow on 25 October, the day before the rain, and spread it over the catchment as a depth per hourly step.

The model runs on one-hour steps (`dt_seconds: 3600`), routes runoff to the outlet by its travel time through the flow network (`routing_method: width_function`, with velocities from the drainage area, `velocity_mode: hydraulic_geometry`), and uses D-infinity flow routing for the TWI.

In [ ]:
pre_storm_flow = float(observed.loc[observed.date == storm_start, "flow_m3s"].iloc[0])
q0 = pre_storm_flow * 3600 / catchment_area_m2           # metres per hourly step
print(f"Gauged flow on {storm_start}: {pre_storm_flow} m3/s -> q0 = {q0:.2e} m per hour")
n_hours = len(rain)


def write_topmodel_param(fname, write_fname, n_steps):
    """Parameter file for a TOPMODEL run of the storm, stopping after n_steps hours."""
    with open(fname, "w") as f:
        f.write("read_fname: ettrick_catchment_DEM.tif\n")
        f.write(f"write_fname: {write_fname}\n")
        f.write("write_output_format: gtiff\n")
        # The model, its time step and forcing
        f.write("column_model: topmodel\n")
        f.write("saturated_zone_mode: index\n")
        f.write("rainfall_fname: storm_rainfall.csv\n")
        f.write("dt_seconds: 3600\n")
        f.write(f"n_timesteps: {n_steps}\n")
        # Routing
        f.write("flow_method: dinf\n")
        f.write("twi_window_radius_metres: 45\n")
        f.write("routing_method: width_function\n")
        f.write("velocity_mode: hydraulic_geometry\n")
        # TOPMODEL parameters and starting state
        f.write("szm_metres: 0.02\n")
        f.write("ln_t0: 4.0\n")
        f.write("td_hours_per_metre: 50\n")
        f.write("srmax_metres: 0.05\n")
        f.write("sr0_metres: 0.002\n")
        f.write(f"q0_metres_per_step: {q0:.6g}\n")
        # Outputs (the rasters are the state at the END of the run)
        f.write("write_saturation_raster: true\n")
        f.write("write_water_table_raster: true\n")
        f.write("write_twi_raster: true\n")


write_topmodel_param("storm.param", "storm", n_hours)

## Step 5: Run the storm

In [ ]:
import time

started = time.time()
!lsdtt-distributed-hydrology ./ storm.param
print(f"\nThe whole storm ran in {time.time() - started:.1f} s")

### The wetness index

The TWI the model computed from the DEM: high (blue) along the valley floors and in hollows, where water converges onto gentle slopes; low on the steep, convex hillsides.

In [ ]:
def read_raster(fname):
    with rasterio.open(fname) as src:
        data = src.read(1).astype(float)
        data[data == src.nodata] = np.nan
        left, bottom, right, top = src.bounds
    return data, [left / 1000, right / 1000, bottom / 1000, top / 1000]   # extent in km


hillshade, extent = read_raster("ettrick_hs_hillshade.tif")
twi, _ = read_raster("storm_twi.tif")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(hillshade, cmap="gray", extent=extent)
im = ax.imshow(twi, cmap="YlGnBu", extent=extent, alpha=0.85, vmin=1, vmax=10)
fig.colorbar(im, ax=ax, shrink=0.7, label="Topographic wetness index, ln(a / tan β)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

## Step 6: The hydrograph

`storm_discharge.csv` has one row per hour: rainfall, the runoff components, the mean deficit, `contributing_area_fraction` and the discharge at the outlet. The contributing area is the part of the catchment **producing saturation-excess runoff** that hour: saturated ground being rained on. It drops to zero in a dry hour even though the ground is still saturated; the maps in Step 7 show the saturated area itself. The NRFA only has daily means, so we show them as steps across each day.

In [ ]:
q = pd.read_csv("storm_discharge.csv")
q["step_start"] = rain["time"].iloc[:len(q)].to_numpy()
q["time"] = q["step_start"] + pd.Timedelta(hours=1)        # plotted at the end of each hourly step

fig, (ax_r, ax_q, ax_s) = plt.subplots(3, 1, figsize=(11, 8), sharex=True,
                                       gridspec_kw={"height_ratios": [1, 3, 1.5]})
ax_r.bar(rain.time, rain.rainfall_mm, width=1 / 24, color="tab:blue")
ax_r.invert_yaxis()
ax_r.set_ylabel("Rain (mm/h)")

day_obs = observed[(observed.date >= storm_start) & (observed.date <= storm_end)]
ax_q.step(day_obs.date, day_obs.flow_m3s, where="post", color="black", lw=1.5, label="NRFA daily mean (observed)")
ax_q.plot(q.time, q.discharge_m3_per_second, color="tab:orange", label="TOPMODEL (hourly)")
ax_q.set_ylabel("Flow (m³ s⁻¹)")
ax_q.legend(loc="upper left")

ax_s.plot(q.time, 100 * q.contributing_area_fraction, color="tab:green")
ax_s.set_ylabel("Contributing\narea (%)")
plt.tight_layout()
plt.show()

print(f"Peak flow {q.discharge_m3_per_second.max():.1f} m3/s at {q.time[q.discharge_m3_per_second.idxmax()]}")
print(f"Largest contributing area {100 * q.contributing_area_fraction.max():.1f}% of the catchment")
daily_sim = q.groupby(q.step_start.dt.date).discharge_m3_per_second.mean().round(1)
print(pd.DataFrame({"simulated (m3/s)": daily_sim.to_numpy(),
                    "observed (m3/s)": day_obs.flow_m3s.to_numpy()[:len(daily_sim)]},
                   index=[str(d) for d in daily_sim.index]))

**Reading it.** The simulated flood is about the right size, and recedes much like the river, but it peaks about a day **late**: the river was already high on 26–27 October, and the reanalysis rainfall arrives mostly late on the 27th. That is the forcing, not the model (Step 3).

Notice how **small the contributing area** stays: a percent or two of the catchment, along the streams. Most of the flood is **subsurface flow**: in TOPMODEL the outflow from the saturated zone grows exponentially as the mean deficit falls, so a wet catchment can deliver a large flood through the soil, with saturated ground only along the valley floors.

## Step 7: The catchment wetting up

The model writes its maps once, at the **end** of a run. To see the state during the storm we run it again, stopped at the hours we want (`n_timesteps`): the model is deterministic, so a run stopped at hour 60 is exactly the first 60 hours of the full run. Each run takes a fraction of a second.

For each time we map the **saturation deficit** (`_water_table.tif`, in metres): how much water each pixel needs before it saturates, from brown (dry) to blue (wet). Navy pixels are **saturated** (`_saturation.tif`). Watch the deficit fall everywhere as the rain soaks in, and the saturated cells spread along the streams.

In [ ]:
slice_hours = [24, 60, 84, 132]        # before the rain, during it, near the peak, the recession
for h in slice_hours:
    write_topmodel_param(f"slice_{h}.param", f"slice_{h}", h)
    !lsdtt-distributed-hydrology ./ slice_{h}.param > slice_{h}.log

from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(2, 2, figsize=(11, 11), sharex=True, sharey=True)
for ax, h in zip(axes.ravel(), slice_hours):
    deficit, _ = read_raster(f"slice_{h}_water_table.tif")
    saturated, _ = read_raster(f"slice_{h}_saturation.tif")
    ax.imshow(hillshade, cmap="gray", extent=extent)
    im = ax.imshow(deficit, cmap="BrBG_r", extent=extent, vmin=0.05, vmax=0.2, alpha=0.85)
    ax.imshow(np.where(saturated == 1, 1.0, np.nan), cmap=ListedColormap(["navy"]), extent=extent)
    when = rain.time.iloc[0] + pd.Timedelta(hours=h)
    ax.set_title(f"hour {h}: {when:%d %b %H:%M}, {100 * np.nanmean(saturated):.1f}% saturated")
fig.colorbar(im, ax=axes, shrink=0.6, extend="both", label="Saturation deficit (m); navy = saturated")
plt.show()

**What the maps show.** Before the rain (hour 24) the deficit is lowest along the valley floors and highest on the slopes and ridges: the TWI pattern. As the storm soaks in, the deficit falls everywhere and the saturated cells spread along the stream network, from under half a percent of the catchment to over one percent near the peak; in the recession the catchment starts to dry and the saturated area shrinks again.

This is the **spatial** behaviour S-HYPE could not give us: the wet ground is in the right places because water drains to it from upslope.

## Things to try

* **Change `ln_t0`.** A less transmissive soil (lower `ln_t0`, e.g. 3) drains more slowly, so more of the catchment saturates.
* **Change `szm_metres`.** Larger `m` (0.04) gives a slower, longer flood; smaller gives a flashier one.
* **Another storm.** The other large floods in 2021–2024 were 21 January 2024, 27 December 2023 and 30 December 2022. In January 2024 the temperatures fell below freezing: TOPMODEL has no snow, so some of that rain was really snow.
* **Dynamic saturated zone.** `saturated_zone_mode: dynamic` gives every pixel its own deficit, moved down the flow network, instead of deriving it from the catchment mean: pixels wet and dry at their own pace. It needs `initial_deficit_metres` and, for stable steps, `saturated_zone_substeps: 0`. It takes a few times longer.
* **Shorter steps.** `dt_seconds: 900` (15 minutes) with 15-minute rainfall, if you have it.

## Data and references

* Daily flow and station data: UK National River Flow Archive, station 21017 (https://nrfa.ceh.ac.uk/data/station/info/21017). Data © UKCEH, used under the NRFA terms.
* Weather: Open-Meteo historical weather API (https://open-meteo.com), based on ERA5 and other reanalysis and model data (CC BY 4.0).
* DEM: Copernicus GLO-30 (© DLR e.V. 2010–2014 and © Airbus Defence and Space GmbH 2014–2018, provided under COPERNICUS by the European Union and ESA).
* Beven, K. J. and Kirkby, M. J. (1979). A physically based, variable contributing area model of basin hydrology. *Hydrological Sciences Bulletin*, 24(1), 43–69. https://doi.org/10.1080/02626667909491834
* The lsdtt3 documentation of the model: https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/hydrological_modelling.html, and every input: https://lsdtopotools.github.io/lsdtt3/cli_reference/cli_reference_lsdtt_distributed_hydrology.html